# Supplementary condition: trimming by voice activity detection (WavLM Base)

**Not in the analysis plan and not pre-registered** (supplementary analysis): a
sensitivity analysis of the trimming rule for WavLM Base, layers 3 to 5. No
unit of the replication rule and no table of the plan depends on it. The
definition was fixed by the first author before this notebook was written
(`docs/decisions.md`, Section 4).

The condition `vad`: the leading and trailing non-speech found by the WebRTC
voice activity detector (`webrtcvad-wheels`, pinned in `requirements.txt`)
is removed. The detector judges frames of 10 ms of the mono 16 kHz waveform
at aggressiveness 3; the kept audio starts with the first run of 3
consecutive speech frames and ends with the last one. Pauses inside the
utterance are kept. A file without such a run, or with fewer than 0.5 s
left, keeps its untrimmed audio and is flagged, as in the trimmed
conditions. The rule is in the experiment configuration
(`vad_conditions`); nothing of it is set in this notebook.

Runtime: T4. Steps, each resumable (run the notebook again from the top at
the same commit after a disconnect):

1. the sha256 of every file under `runs/`, `tables/` and `checks/` is
   recorded;
2. the file lists against the shards of the untrimmed condition; the
   encoder; a short check of 25 files per split with the time and the size
   of the extraction, and the fallback files among them;
3. with `EXTRACT = True`: the unpadded extraction of all 12 layers into
   `features/wavlm-base/unpadded/vad/` (about 11.3 GB, about 1 h; finished
   shards are kept), then the check of every file against the rule;
4. the 30 models trained on untrimmed audio, trained again because model
   weights are not stored, and scored on untrimmed and on `vad` audio. A
   hard stop: their scores on untrimmed audio must equal those of the
   existing runs within 1e-4, with the same selected epoch;
5. the 30 models trained on `vad` audio, scored on `vad` and on untrimmed
   audio; then every score set against the file lists, and the run records;
6. the tables, under `tables/wavlm-base/vad/`: the six effects with their
   four cells and file- and speaker-level intervals, the same next to the
   condition trimmed at 40 dB, the audio removed per split and class, the
   fallback counts and the per-attack effect. The cell trained and tested
   on untrimmed audio is read from the existing runs;
7. a hard stop unless every file of step 1 is unchanged and no file was
   added outside the folder of the tables and the check tables of this
   notebook (`vad_...`).

The runs are written to `runs_vad/wavlm-base/unpadded/`, not to `runs/`:
the models trained again have the run ids of existing runs. Finished runs
are skipped.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run does

`EXTRACT` is the only value to set. As saved, the notebook stops after the
short check; nothing is extracted or trained.

In [ ]:
from ssl_silence import analysis, shards, training, vad

EXTRACT = False    # False: the short check and the estimate only; True: then the extraction, the runs and the tables

ENCODER, CONDITION = vad.ENCODER, vad.CONDITION
RULE = cfg.vad_rule(CONDITION)
FEATURES = cfg.feature_dir(ENCODER, "unpadded", CONDITION)
RUNS = cfg.vad_runs_dir(ENCODER, "unpadded")              # the runs of this condition
EXISTING_RUNS = cfg.runs_dir(ENCODER, "unpadded")         # only read
TABLES = cfg.vad_tables_dir(ENCODER)
planned = vad.planned_runs()
LAYERS = vad.layers(planned)

# Nothing that exists is written. The sha256 of every file under these folders is recorded now and
# compared in the last cell; the tables of this condition and the check tables of this notebook
# (their names start with "vad_") are left out.
GUARDED = [os.path.join(cfg.ROOT, "runs"), os.path.join(cfg.ROOT, "tables"), CHECKS]
OWN = ("vad_",)
OWN_FOLDERS = (TABLES,)
assert not RUNS.startswith(GUARDED[0] + os.sep), RUNS
BEFORE = verify.file_hashes(GUARDED, skip=OWN, skip_folders=OWN_FOLDERS)
assert BEFORE, "no file under runs/, tables/ and checks/: the Drive root is not the one of the analysis"
print("this run:", ENCODER, "| condition:", CONDITION, "| rule:", RULE, "| layers:", LAYERS,
      "| runs planned:", len(planned), "| extraction, runs and tables:", EXTRACT)
print("existing files under runs/, tables/ and checks/:", len(BEFORE),
      "| their sha256 is recorded and must not change")

### File lists

The datasets are unpacked to the local disk (a few minutes). Every split
must have the files of the shards of the untrimmed condition in the same
order with the same labels: a hard stop.

In [ ]:
from ssl_silence import datasets, extraction

datasets.ensure_asvspoof(cfg.la_zip())
datasets.ensure_itw(cfg.itw_zip(), sha256=cfg.ITW_ZIP_SHA256)
file_lists = {}
for split in cfg.SPLITS:
    file_lists[split] = datasets.items(split)
    print(split, datasets.check_items(file_lists[split], split))
sizes = {split: len(file_lists[split]) for split in cfg.SPLITS}
reference = shards.shard_lengths(cfg.feature_dir(ENCODER, "unpadded", vad.BASE))
record("vad_file_lists", pd.concat([extraction.file_list_check(file_lists[split], reference, split)
                                    for split in cfg.SPLITS], ignore_index=True))

### Encoder and short check

25 files per split go through the rule and the encoder. A hard stop if a
kept interval is not what the rule allows, or a frame count is not the
output length of the convolutional encoder. The fallback files among the
25 are shown per split and class: if most files of a class fall back, stop
here and do not extract.

In [ ]:
import torch

from ssl_silence import encoders

assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again from the top"
DEVICE = "cuda"
entry = cfg.encoder(ENCODER)
model = encoders.load_encoder(ENCODER, DEVICE)
described = encoders.describe(model)
print("encoder:", ENCODER, "|", entry["hf_id"], "| revision:", entry["revision"])
record("vad_encoder", pd.DataFrame([{
    "encoder": ENCODER, "hf_id": entry["hf_id"], "revision": entry["revision"], "git_commit": HEAD,
    "gpu": provenance.gpu_name(DEVICE), **{key: str(value) for key, value in described.items()},
    **provenance.versions(), "webrtcvad-wheels": provenance.installed_version("webrtcvad-wheels"),
    **{f"vad_{key}": value for key, value in RULE.items()}}]), stop=False)

In [ ]:
short = pd.concat([vad.short_check(file_lists[split], split, model=model, device=DEVICE)
                   for split in cfg.SPLITS], ignore_index=True)
record("vad_short_check", short, stop=False)
estimate = extraction.extraction_estimate(short, sizes, entry["n_layers"], entry["hidden_size"])
record("vad_estimate", estimate, stop=False)
verify.require_ok(short, "short check of the condition " + CONDITION)
assert len(short) == len(cfg.SPLITS) * cfg.SHORT_CHECK_FILES, len(short)
display(short.assign(removed_s=(short["n_samples"] - short["kept_end"] + short["kept_start"]) / cfg.SAMPLE_RATE)
        .groupby(["split", "label"]).agg(files=("ok", "size"), fallback=("trim_fallback", "sum"),
                                         median_removed_s=("removed_s", "median")))
print("short check passed:", CONDITION, "| files encoded:", len(short),
      "| fallback files:", int(short["trim_fallback"].sum()))
for row in estimate.itertuples():
    print(f"expected for {row.condition}: {row.hours:.1f} h on {provenance.gpu_name(DEVICE)}, "
          f"{row.feature_gb:.1f} GB in {row.n_shards} shards")

### Extraction

All 12 layers, in the format of the other conditions; the shards also store
the samples of every file and the interval that was kept. Then, as a hard
stop: every file of the shards has the id and the label of the untrimmed
shards, an interval inside the file at frame boundaries, at least 0.5 s
kept or the fallback flag, and the number of real samples of that interval;
and every shard records the encoder, the condition, a clean commit and the
library versions of this runtime.

In [ ]:
if EXTRACT:
    print("free on Google Drive: %.1f GB" % cfg.drive_quota()["free_gb"], "| writing to", FEATURES)
    for split in cfg.SPLITS:
        paths = extraction.extract_shards(file_lists[split], FEATURES, split, encoder=ENCODER,
                                          extraction="unpadded", condition=CONDITION, device=DEVICE, model=model)
        print(split, "| shards:", len(paths), "of", shards.n_shards(sizes[split], cfg.SHARD_SIZE))
else:
    print("EXTRACT is False: nothing is extracted or trained in this run")

In [ ]:
if EXTRACT:
    kept = shards.kept_intervals(FEATURES)
    files = vad.shards_check(kept, reference, RULE)
    record("vad_shards", files, stop=False)
    shard_meta = extraction.shard_meta_check(extraction.shard_meta_table(FEATURES), ENCODER, CONDITION)
    record("vad_shard_meta", shard_meta, stop=False)
    verify.require_ok(files, "the files of the shards against the rule")
    verify.require_ok(shard_meta, "what the shards record")
    assert int(shard_meta["n_files"].sum()) == sum(sizes.values()), "not every file is in a shard"
    del model
    torch.cuda.empty_cache()
    print("extraction finished:", CONDITION, "| files:", int(shard_meta["n_files"].sum()),
          "| shards:", int(shard_meta["n_shards"].sum()), "| fallback files:", int(files["n_fallback"].sum()))

### The models trained on untrimmed audio, trained again

30 runs (layers 3 to 5, both poolings, five seeds), scored on untrimmed and
on `vad` audio. A hard stop: every score set on untrimmed audio equals that
of the existing run with the same id within 1e-4, and the selected epoch is
the same. The models of the new condition are trained only after this.

In [ ]:
if EXTRACT:
    clean_runs, vad_runs = vad.runs_of(planned, vad.BASE), vad.runs_of(planned, CONDITION)
    status = training.run_experiments(clean_runs, runs_dir=RUNS, device=DEVICE)
    record("vad_training_clean", status, stop=False)
    clean_scores = vad.clean_scores_check(planned, RUNS, EXISTING_RUNS)
    record("vad_clean_scores", clean_scores)          # a hard stop
    print("the clean models trained again have the scores of the existing runs | score sets:", len(clean_scores),
          "| exactly equal:", int(clean_scores["exactly_zero"].sum()),
          "| largest difference:", float(clean_scores["max_abs_diff"].max()))

### The models trained on `vad` audio

30 runs, scored on `vad` and on untrimmed audio. Then every score set of
the 60 runs against the file lists, and the run records: a hard stop.

In [ ]:
if EXTRACT:
    status = training.run_experiments(vad_runs, runs_dir=RUNS, device=DEVICE)
    record("vad_training", status, stop=False)
    sets = training.score_set_check(planned, RUNS, file_lists)
    records = training.run_record_check(planned, RUNS, n_train=len(file_lists["train"]))
    record("vad_score_sets", sets, stop=False)
    record("vad_run_records", records, stop=False)
    verify.require_ok(sets, "vad_score_sets")
    verify.require_ok(records, "vad_run_records")
    print("training finished:", CONDITION, "| runs:", len(records), "| score sets checked:", len(sets),
          "| commits:", sorted(set(records["git_commit"])))

### Tables

Written under `tables/wavlm-base/vad/` with the functions, resamples and
seeds of the analysis:

- `effects_2x2.csv`: the six effects per test set, pooling and layer with
  file- and speaker-level intervals, and the four cells (`eer_cc`, `eer_ct`,
  `eer_tc`, `eer_tt`: train then test, c untrimmed, t `vad`);
- `cells_2x2.csv`: the four cells alone;
- `effects_vad_and_trim40.csv`: every row of the effects next to the row of
  the condition trimmed at 40 dB, read from `tables/wavlm-base/effects_2x2.csv`;
- `removed_amount.csv`: seconds removed per split and class;
- `fallback_counts.csv`: files that kept their untrimmed audio;
- `per_attack_effect.csv`: `vad` minus untrimmed per attack, matched cells.

In [ ]:
if EXTRACT:
    tables = analysis.ScoreTables(
        ENCODER, RUNS, TABLES, speaker_maps={name: datasets.speaker_map(file_lists[name]) for name in cfg.TEST_SETS},
        attack_map=datasets.attack_map(file_lists["eval"]), conditions=(CONDITION,), layers=LAYERS,
        cross_layers={CONDITION: LAYERS}, detail_layers=LAYERS, base_runs_dir=EXISTING_RUNS)
    effects = tables.effects()
    cells = tables.write("cells_2x2", vad.cells_from_effects(effects))
    reference_effects = analysis.read_table(os.path.join(cfg.tables_dir(ENCODER), "effects_2x2.csv"))
    side = tables.write("effects_vad_and_trim40", vad.side_by_side(effects, reference_effects))
    removed = tables.write("removed_amount", vad.summarise_removed(vad.removed_per_file(kept)))
    fallback = tables.fallback_counts({CONDITION: FEATURES})
    attacks = tables.per_attack_effect()
    record("vad_steps", tables.steps_table(), stop=False)
    display(cells)
    display(removed)
    display(fallback)
    print("tables written to", TABLES, "| effects:", len(effects), "rows | side by side:", len(side),
          "rows | per attack:", len(attacks), "rows")

### Nothing that existed was changed

A hard stop unless every file that existed under `runs/`, `tables/` and
`checks/` at the start has the same sha256 and no file was added there,
except the tables of this condition and the check tables of this notebook.

In [ ]:
unchanged = verify.unchanged_check(BEFORE, GUARDED, skip=OWN, skip_folders=OWN_FOLDERS)
record("vad_files_unchanged", unchanged)          # a hard stop
print("check tables written to", CHECKS)
print("vad run finished:", ENCODER, "| extraction, runs and tables:", EXTRACT,
      "| files that existed at the start:", len(BEFORE), "unchanged")